# Mixing materials

To simplify a neutronics model it is common to combine regions and their materials. For example, cooling channels are often combined with the structural material which reduces the complexity of the model and makes the simulation faster. 

There are several different ways of making mixed materials based on the methods we have seen in Parts 1 to 4.

| Material type | Combinations | Mixing Method |
|:-:|:-:|:-:|
| code block 1 | openmc.Material() + openmc.Material() | openmc.Material.mix_materials() |
| code block 2 | openmc.Material() + nmm.Material() | openmc.Material.mix_materials() |
| code block 3 | nmm.Material() + nmm.Material() | openmc.Material.mix_materials() |
| code block 4 | openmc.Material() + openmc.Material() | nmm.MultiMaterial() |
| code block 5 | openmc.Material() + nmm.Material() | nmm.MultiMaterial() |
| code block 6 | nmm.Material() + nmm.Material() | nmm.MultiMaterial() |

Personally I use a combination of these depending on the task. The benefit of using the neutronics_material_maker is that we can use a standard material definition and density is calculated automatically (as a function of pressure, temperture and enrichment), but this is not always required.

This python notebook allows users to make mixed materials using OpenMC and the Neutronics Material Maker.

First import packages needed and configure the OpenMC nuclear data path

In [1]:
import neutronics_material_maker as nmm
import openmc
from pathlib import Path
# Setting the cross section path to the correct location in the docker image.
# If you are running this outside the docker image you will have to change this path to your local cross section path.
openmc.config['cross_sections'] = Path.home() / 'nuclear_data' / 'cross_sections.xml'

## Code block 1

This first code block is an example of making a mixed material using the native OpenMC mix_materials method with two OpenMC materials.

| Material type | Combinations | Mixing Method |
|:-:|:-:|:-:|
| code block 1 | openmc.Material() + openmc.Material() | openmc.Material.mix_materials() |

In [2]:
steel_mat = openmc.Material()
steel_mat.add_element('Fe', 0.975)
steel_mat.add_element('C', 0.025)
steel_mat.set_density('g/cm3', 7.7)

h20_mat = openmc.Material()
h20_mat.add_elements_from_formula('H2O')
h20_mat.set_density('g/cm3', 1.)

mixed_mat = openmc.Material.mix_materials(
    materials=[
        steel_mat,
        h20_mat,
        ],
    fracs=[0.7, 0.2], # list of combination fractions for each material
    percent_type='vo') # combination fraction type is by volume

mixed_mat

/home/jon/.neutronicsworkshop_3.12/lib/python3.12/site-packages/openmc/material.py:1604: UserWarning: Warning: sum of fractions do not add to 1, void fraction set to 0.10000000000000009
  warnings.warn(msg)


Material
	ID             =	3
	Name           =	(0.7)-(0.2)
	Temperature    =	None
	Density        =	5.59 [g/cm3]
	Volume         =	None [cm^3]
	Depletable     =	False
	S(a,b) Tables  
	Nuclides       
	Fe54           =	0.0425829838055608 [ao]
	Fe56           =	0.6684617786305264 [ao]
	Fe57           =	0.015437697636267471 [ao]
	Fe58           =	0.002054474154519786 [ao]
	C12            =	0.018473492360756644 [ao]
	C13            =	0.0002069418501888542 [ao]
	H1             =	0.1684955087967603 [ao]
	H2             =	2.6245578026329275e-05 [ao]
	O16            =	0.08406038685621364 [ao]
	O17            =	3.1934872454022075e-05 [ao]
	O18            =	0.0001685554587256616 [ao]

## Code block 2

This code block is an example of making a mixed material using the native OpenMC mix_materials method with an OpenMC material and a neutronics_material_maker material.

| Material type | Combinations | Mixing Method |
|:-:|:-:|:-:|
| code block 2 | openmc.Material() + nmm.Material() | openmc.Material.mix_materials() |

In [3]:
steel_mat = openmc.Material()
steel_mat.add_element('Fe', 0.975)
steel_mat.add_element('C', 0.025)
steel_mat.set_density('g/cm3', 7.7)

h20_mat = nmm.Material.from_library(
    'H2O',
    temperature=500,
    pressure=80000
).openmc_material

mixed_mat = openmc.Material.mix_materials(
    materials=[                 # list of neutronics materials
        steel_mat,
        h20_mat],
    fracs=[0.7, 0.3],           # list of combination fractions for each neutronics material
    percent_type='vo')          # combination fraction type

mixed_mat

Material
	ID             =	7
	Name           =	(0.7)-H2O(0.3)
	Temperature    =	None
	Density        =	5.390104347049642 [g/cm3]
	Volume         =	None [cm^3]
	Depletable     =	False
	S(a,b) Tables  
	Nuclides       
	Fe54           =	0.0569786931277138 [ao]
	Fe56           =	0.8944436286125326 [ao]
	Fe57           =	0.02065660406118487 [ao]
	Fe58           =	0.002749014792474815 [ao]
	C12            =	0.024718687094051457 [ao]
	C13            =	0.00027690112630511 [ao]
	H1             =	0.00011762913474324992 [ao]
	H2             =	1.8322414977822388e-08 [ao]
	O16            =	5.8683763399332715e-05 [ao]
	O17            =	2.2294193131484153e-08 [ao]
	O18            =	1.1767098664965938e-07 [ao]

## Code block 3

This next code block is an example of making a mixed material using the native OpenMC mix_materials method with two neutronics_material_maker materials.

| Material type | Combinations | Mixing Method |
|:-:|:-:|:-:|
| code block 3 | nmm.Material() + nmm.Material() | openmc.Material.mix_materials() |

In [4]:
steel_mat = nmm.Material.from_library('a36_steel').openmc_material

h20_mat = nmm.Material.from_library(
    'H2O',
    temperature=500,
    pressure=80000
).openmc_material

mixed_mat = openmc.Material.mix_materials(
    materials=[                 # list of neutronics materials
        steel_mat,
        h20_mat],
    fracs=[0.7, 0.3],           # list of combination fractions for each neutronics material
    percent_type='vo')          # combination fraction type

mixed_mat

Material
	ID             =	12
	Name           =	a36_steel(0.7)-H2O(0.3)
	Temperature    =	None
	Density        =	5.495104347049642 [g/cm3]
	Volume         =	None [cm^3]
	Depletable     =	False
	S(a,b) Tables  
	Nuclides       
	C12            =	0.011266737381450396 [ao]
	C13            =	0.0001262110830901805 [ao]
	Cu63           =	0.0011912780707439364 [ao]
	Cu65           =	0.0005314667893340626 [ao]
	Fe54           =	0.056659775586578524 [ao]
	Fe56           =	0.8894373052473783 [ao]
	Fe57           =	0.02054098622206329 [ao]
	Fe58           =	0.002733628180567176 [ao]
	Mn55           =	0.010262271370285656 [ao]
	P31            =	0.0007068796671723824 [ao]
	S32            =	0.0008112285058568277 [ao]
	S33            =	6.390508639241955e-06 [ao]
	S34            =	3.581523774215343e-05 [ao]
	S36            =	1.2444885861991972e-07 [ao]
	Si28           =	0.005086935506938584 [ao]
	Si29           =	0.00025830007096061156 [ao]
	Si30           =	0.00017027256106277844 [ao]
	H1            

## Code block 4

This code block is an example of making a mixed material using the neutronics_material_maker MultiMaterial class with two OpenMC materials.

| Material type | Combinations | Mixing Method |
|:-:|:-:|:-:|
| code block 4 | openmc.Material() + openmc.Material() | nmm.MultiMaterial() |

In [5]:
steel_mat = openmc.Material()
steel_mat.add_element('Fe', 0.975)
steel_mat.add_element('C', 0.025)
steel_mat.set_density('g/cm3', 7.7)

h20_mat = openmc.Material()
h20_mat.add_elements_from_formula('H2O')
h20_mat.set_density('g/cm3', 1.)

mixed_mat = nmm.Material.from_mixture(
    materials=[
        steel_mat,
        h20_mat
    ],
    fracs=[0.7, 0.3],           # list of combination fractions for each neutronics material
    percent_type='vo')          # combination fraction type

mixed_mat.openmc_material   # returns the OpenMC-compatible version of the nmm.MultiMaterial

Material
	ID             =	17
	Name           =	
	Temperature    =	None
	Density        =	5.689999999999999 [g/cm3]
	Volume         =	None [cm^3]
	Depletable     =	False
	S(a,b) Tables  
	Nuclides       
	C12            =	0.0164005990652958 [ao]
	C13            =	0.00018372109877760513 [ao]
	Fe54           =	0.03780478703001352 [ao]
	Fe56           =	0.5934543078104124 [ao]
	Fe57           =	0.013705448026791902 [ao]
	Fe58           =	0.0018239435316447926 [ao]
	H1             =	0.2243831780786387 [ao]
	H2             =	3.495087940392556e-05 [ao]
	O16            =	0.11194207423099993 [ao]
	O17            =	4.252723543754908e-05 [ao]
	O18            =	0.00022446301258383423 [ao]

## Code block 5

The next code block is an example of making a mixed material using the neutronics_material_maker MultiMaterial class with an OpenMC and a neutronics_material_maker material.

| Material type | Combinations | Mixing Method |
|:-:|:-:|:-:|
| code block 5 | openmc.Material() + nmm.Material() | nmm.MultiMaterial() |

In [6]:
steel_mat = openmc.Material()
steel_mat.add_element('Fe', 0.975)
steel_mat.add_element('C', 0.025)
steel_mat.set_density('g/cm3', 7.7)

h20_mat = nmm.Material.from_library(
    'H2O',
    temperature=700,
    pressure=80000
).openmc_material

mixed_mat = nmm.Material.from_mixture(
    name='mixed_material',  # name of homogeneous material
    materials=[
        steel_mat,
        h20_mat
    ],
    fracs=[0.7, 0.3],           # list of combination fractions for each neutronics material
    percent_type='vo')          # combination fraction type

mixed_mat.openmc_material   # returns the OpenMC-compatible version of the nmm.MultiMaterial

Material
	ID             =	23
	Name           =	mixed_material
	Temperature    =	None
	Density        =	5.390074355138227 [g/cm3]
	Volume         =	None [cm^3]
	Depletable     =	False
	S(a,b) Tables  
	Nuclides       
	C12            =	0.024719940942951227 [ao]
	C13            =	0.0002769151720418938 [ao]
	Fe54           =	0.05698158335693256 [ao]
	Fe56           =	0.8944889990302808 [ao]
	Fe57           =	0.020657651861991467 [ao]
	Fe58           =	0.0027491542355269436 [ao]
	H1             =	8.382387675267346e-05 [ao]
	H2             =	1.3056764025890751e-08 [ao]
	O16            =	4.181872595854484e-05 [ao]
	O17            =	1.5887098901414527e-08 [ao]
	O18            =	8.385370090340267e-08 [ao]

## Code block 6

Finally, this last code block is an example of making a mixed material using the neutronics_material_maker MultiMaterial class with two neutronics_material_maker materials.

| Material type | Combinations | Mixing Method |
|:-:|:-:|:-:|
| code block 6 | nmm.Material() + nmm.Material() | nmm.MultiMaterial() |

In [7]:
steel_mat = nmm.Material.from_library('a36_steel').openmc_material

h20_mat = nmm.Material.from_library(
    name='H2O',
    temperature=500,
    pressure=80000
).openmc_material

mixed_mat = nmm.Material.from_mixture(
    name='mixed_material',  # name of homogeneous material
    materials=[
        steel_mat,
        h20_mat
    ],
    fracs=[0.7, 0.3],           # list of combination fractions for each neutronics material
    percent_type='vo')          # combination fraction type

mixed_mat.openmc_material

Material
	ID             =	30
	Name           =	mixed_material
	Temperature    =	None
	Density        =	5.495104347049641 [g/cm3]
	Volume         =	None [cm^3]
	Depletable     =	False
	S(a,b) Tables  
	Nuclides       
	C12            =	0.011266737381450396 [ao]
	C13            =	0.0001262110830901805 [ao]
	Cu63           =	0.0011912780707439364 [ao]
	Cu65           =	0.0005314667893340626 [ao]
	Fe54           =	0.056659775586578524 [ao]
	Fe56           =	0.8894373052473783 [ao]
	Fe57           =	0.02054098622206329 [ao]
	Fe58           =	0.002733628180567176 [ao]
	H1             =	0.00011624426748258767 [ao]
	H2             =	1.8106702155531904e-08 [ao]
	Mn55           =	0.010262271370285656 [ao]
	O16            =	5.799286974580399e-05 [ao]
	O17            =	2.2031719908008832e-08 [ao]
	O18            =	1.1628562665958013e-07 [ao]
	P31            =	0.0007068796671723824 [ao]
	S32            =	0.0008112285058568277 [ao]
	S33            =	6.390508639241955e-06 [ao]
	S34            =	3.58

**Learning Outcomes for Part 5:**

- Mixed materials can be created using in-built OpenMC features or the neutronics material maker.
- The mix fraction of each material will have an impact on the overall properties of the mixed material.